In [1]:
# !pip install unsloth unsloth_zoo vllm trl transformers datasets accelerate peft \
#             bitsandbytes sentence-transformers rank-bm25 openai anthropic \
#             tenacity omegaconf rich jsonlines tqdm


In [2]:
import sys
sys.path.insert(0, "..")   # project root

import os
import json
from pathlib import Path
from src.data.excel_parser      import parse_telecom_functions, load_function_schema
from src.data.dataset_generator import TelcoDatasetGenerator
from src.data.retrieval          import FunctionRetriever


In [3]:
# Cell 3 — Load your function_schema.json
# ⚠️  Replace this path once you provide the schema file
SCHEMA_PATH = "../data/processed/function_schema.json"

# If schema doesn't exist yet, create a minimal example for testing:
if not Path(SCHEMA_PATH).exists():
    example_schema = {
        "get_cell_kpis": {
            "name": "get_cell_kpis",
            "description": "Retrieve KPI metrics for a specific cell tower",
            "parameters": {
                "cell_id":   {"type": "string", "required": True,  "description": "Cell tower ID"},
                "metric":    {"type": "string", "required": True,  "description": "KPI metric name"},
                "start_time":{"type": "string", "required": True,  "description": "ISO datetime start"},
                "end_time":  {"type": "string", "required": True,  "description": "ISO datetime end"},
            },
            "examples": ["What is the RSRP for cell 123 this morning?"],
            "domain":   {"category": "performance_management"},
            "constraints": {},
            "tags": ["kpi", "performance"],
        },
        "set_handover_threshold": {
            "name": "set_handover_threshold",
            "description": "Configure the handover threshold for a cell",
            "parameters": {
                "cell_id":    {"type": "string", "required": True},
                "threshold_db": {"type": "number", "required": True, "description": "dB value"},
            },
            "examples": ["Set handover threshold to -3 dB for cell 456"],
            "domain":   {"category": "configuration"},
            "constraints": {"threshold_db": {"min": -20, "max": 0}},
            "tags": ["configuration", "handover"],
        },
    }
    Path(SCHEMA_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(SCHEMA_PATH, "w") as fh:
        json.dump(example_schema, fh, indent=2)
    print(f"Example schema written to {SCHEMA_PATH}")

function_library = load_function_schema(SCHEMA_PATH)
print(f"Loaded {len(function_library)} functions: {list(function_library.keys())}")


[excel_parser] Loaded function_schema.json  →  26 functions
Loaded 26 functions: ['TRAM_NHA_MANG_KHAC_PROVINCE', 'KE_HOACH_TRIEN_KHAI', 'VUNG_PHU_PROVINCE', 'KQI_PROVINCE', 'SUB_ATTACHED_STATION', 'DOWNLOAD_THROUGHPUT_OSS', 'SPEEDTEST_PROVINCE', 'VUNG_LOM_ALL', 'PAKH_ALL', 'SUB_ATTACHED_ALL', 'REGIONAL_STATION_INFO', 'ALARM_COUNT', 'RADIO_TRAFFIC', 'RADIO_KPI', 'TONG_QUAN_KPI_VIEN_THONG', 'TOP_TRAM_MAX', 'TOP_TRAM_MIN', 'TOP_CELL_MAX', 'TOP_CELL_MIN', 'TOP_SUB_ATTACHED_MAX', 'TOP_SUB_ATTACHED_MIN', 'TOP_PORT', 'ALARM_UNRESOLVED', 'THONG_KE_CNTT', 'THONG_KE_KPI', 'NGUONG_KPI']


In [4]:
# Cell 4 — Configure API provider
# Set your API key as an environment variable:
# os.environ["OPENAI_API_KEY"] = "sk-..."         # OpenAI
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # Anthropic
# os.environ["TOGETHER_API_KEY"] = "..."          # Together AI (cheap!)

# For Together AI (very cost-effective for bulk generation):
# provider = "together"
# model    = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"

provider = "openrouter"
model    = "nvidia/nemotron-3-nano-30b-a3b:free"


In [ ]:
# Cell 5 — Generate dataset
generator = TelcoDatasetGenerator(
    function_library  = function_library,
    provider          = provider,
    model             = model,
    max_workers       = 4,
    requests_per_minute = 100,
    temperature       = 0.9,
    seed              = 42,
)

train_samples, test_samples = generator.generate(
    total                   = 3000,    # ← increase to 2700 for full run
    output_dir              = "../data/processed",
    reserved_test_functions = 7,
    workflow_distribution   = {
        "single_call": 0.60,
        "parallel":    0.20,
        "sequential":  0.15,
        "abstention":  0.05,
    },
)

print(f"\nGenerated: {len(train_samples)} train, {len(test_samples)} test samples")


[DatasetGenerator] Provider=openrouter  Model=nvidia/nemotron-3-nano-30b-a3b:free  Functions=26
[DatasetGenerator] Train functions : 19
[DatasetGenerator] Test  functions : ['DOWNLOAD_THROUGHPUT_OSS', 'SUB_ATTACHED_ALL', 'RADIO_TRAFFIC', 'TOP_TRAM_MIN', 'SPEEDTEST_PROVINCE', 'TOP_SUB_ATTACHED_MIN', 'TOP_SUB_ATTACHED_MAX']
[DatasetGenerator] Workflow counts : {'single_call': 1800, 'parallel': 600, 'sequential': 450, 'abstention': 150}


single_call:   0%|          | 1/360 [00:03<23:41,  3.96s/it]

  [WARN] single_call error for 'KQI_PROVINCE': RetryError[<Future at 0x2885a14f6b0 state=finished raised RuntimeError>]
  [WARN] single_call error for 'RADIO_KPI': RetryError[<Future at 0x2885a397950 state=finished raised RuntimeError>]
  [WARN] single_call error for 'REGIONAL_STATION_INFO': RetryError[<Future at 0x28859f16420 state=finished raised RuntimeError>]
  [WARN] single_call error for 'KE_HOACH_TRIEN_KHAI': RetryError[<Future at 0x2885a151be0 state=finished raised RuntimeError>]


single_call:   1%|▏         | 5/360 [00:07<08:31,  1.44s/it]

  [WARN] single_call error for 'TOP_PORT': RetryError[<Future at 0x2885a107c80 state=finished raised RuntimeError>]
  [WARN] single_call error for 'TOP_CELL_MAX': RetryError[<Future at 0x2885a397b90 state=finished raised RuntimeError>]
  [WARN] single_call error for 'RADIO_KPI': RetryError[<Future at 0x2885a397f20 state=finished raised RuntimeError>]
  [WARN] single_call error for 'TOP_CELL_MIN': RetryError[<Future at 0x2885a397a40 state=finished raised RuntimeError>]


...

  [WARN] single_call error for 'TOP_CELL_MIN': RetryError[<Future at 0x2885a42f950 state=finished raised RuntimeError>]
  [WARN] single_call error for 'KQI_PROVINCE': RetryError[<Future at 0x2885a493200 state=finished raised RuntimeError>]
  [WARN] single_call error for 'KQI_PROVINCE': RetryError[<Future at 0x2885a4c6030 state=finished raised RuntimeError>]
  [WARN] single_call error for 'PAKH_ALL': RetryError[<Future at 0x2885a4c5970 state=finished raised RuntimeError>]


single_call:  51%|█████▏    | 185/360 [03:08<02:55,  1.00s/it]

  [WARN] single_call error for 'PAKH_ALL': RetryError[<Future at 0x2885a397b00 state=finished raised RuntimeError>]
  [WARN] single_call error for 'THONG_KE_KPI': RetryError[<Future at 0x2885a516090 state=finished raised RuntimeError>]
  [WARN] single_call error for 'TONG_QUAN_KPI_VIEN_THONG': RetryError[<Future at 0x2885a2bc920 state=finished raised RuntimeError>]
  [WARN] single_call error for 'TOP_PORT': RetryError[<Future at 0x2885a492090 state=finished raised RuntimeError>]


single_call:  52%|█████▎    | 189/360 [03:12<02:51,  1.00s/it]

  [WARN] single_call error for 'NGUONG_KPI': RetryError[<Future at 0x2885a42fc50 state=finished raised RuntimeError>]
  [WARN] single_call error for 'ALARM_COUNT': RetryError[<Future at 0x2885a4c7020 state=finished raised RuntimeError>]
  [WARN] single_call error for 'TOP_PORT': RetryError[<Future at 0x2885a516120 state=finished raised RuntimeError>]
  [WARN] single_call error for 'ALARM_COUNT': RetryError[<Future at 0x2885a516810 state=finished raised RuntimeError>]


In [ ]:
# Cell 6 — Inspect a sample
import random
sample = random.choice(train_samples)
print(json.dumps({
    "id":            sample.id,
    "query":         sample.query,
    "workflow_type": sample.workflow_type,
    "function_name": sample.function_name,
    "ground_truth":  sample.ground_truth,
    "retrieved_fns": sample.retrieved_functions,
}, indent=2))


In [ ]:
# Cell 7 — Build retrieval index
retriever = FunctionRetriever(
    function_library = function_library,
    method           = "hybrid",
    index_dir        = "../data/processed/retrieval_index",
)

# Test retrieval
query = "Check the signal quality for the base station in sector 3"
results = retriever.retrieve_with_scores(query, k=3)
print("\nRetrieval test:")
for fn, score in results:
    print(f"  {fn}: {score:.4f}")


In [ ]:
# Cell 8 — Save retriever
retriever.save("../data/processed/retrieval_index/retriever.pkl")
print("Retriever saved.")